# DMP Bridge — Run Pipeline

Extract and label a DMP PDF in three steps:

1. **Configure** — choose your PDF, model, and extractor below
2. **Run** — one cell runs the full pipeline
3. **Inspect** — browse the labeled blocks and structured JSON output

In [ ]:
import os
from pathlib import Path

# Navigate to the project root regardless of where Jupyter started.
_cwd = Path.cwd()
if _cwd.name == "notebooks" and (_cwd.parent / "dmpbridge").exists():
    os.chdir(_cwd.parent)
elif not (_cwd / "dmpbridge").exists():
    raise RuntimeError(f"Cannot find project root from {_cwd}.")

print(f"Working directory: {Path.cwd()}")

## 1 — Configuration
Edit the values below, then run all cells.

In [ ]:
#  Input 
PDF_PATH  = Path("data/input/pdfs/sample10.pdf")   # path to your PDF

#  Model ─
MODEL     = "gemma4:e4b"              # options: "llama3.1:8b"  "llama3.3:70b"  "gemma4:e4b"
HOST      = "http://localhost:11434"  # Ollama server URL

# Extractor ─
EXTRACTOR = "pdfplumber"   # options: "pdfplumber"  "docling"

# Annotation rules
APPLY_RULES = True         # backfill empty question texts from section titles

# Output 
OUT_DIR = Path("data/output/pipeline_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stem            = PDF_PATH.stem
LABELED_JSON    = OUT_DIR / f"{stem}_labeled.json"
STRUCTURED_JSON = OUT_DIR / f"{stem}_structured.json"

print(f"PDF       : {PDF_PATH}  {'✓ exists' if PDF_PATH.exists() else '✗ NOT FOUND'}")
print(f"Model     : {MODEL}")
print(f"Extractor : {EXTRACTOR}")
print(f"Rules     : {APPLY_RULES}")
print(f"Output    : {OUT_DIR}/")

In [ ]:
import requests

try:
    r = requests.get(f"{HOST}/api/tags", timeout=5)
    r.raise_for_status()
    available = [m["name"] for m in r.json().get("models", [])]
    loaded    = any(MODEL in m for m in available)
    print(f"Ollama : running at {HOST}  ✓")
    print(f"Model  : {MODEL}  {'✓ ready' if loaded else '✗ not pulled — run: ollama pull ' + MODEL}")
except Exception:
    print(f"Ollama : NOT reachable at {HOST}  ✗")
    print("  → Start Ollama first, then re-run this cell.")
    raise SystemExit("Ollama must be running before you can run the pipeline.")

## 2 — Run the pipeline

In [ ]:
import dmpbridge

blocks = dmpbridge.process_pdf(
    PDF_PATH,
    model=MODEL,
    host=HOST,
    extractor=EXTRACTOR,
    apply_rules=APPLY_RULES,
    output=LABELED_JSON,
    structured_output=STRUCTURED_JSON,
    raw_dir=None,
)

print(f"Done — {len(blocks)} blocks labeled")
print(f"Labeled JSON    → {LABELED_JSON}")
print(f"Structured JSON → {STRUCTURED_JSON}")

## 3 — Labeled blocks
Every text block from the PDF with its assigned label and confidence score.

In [ ]:
import json
from IPython.display import HTML
from collections import Counter

# Load from disk if the pipeline wasn't run this session
if "blocks" not in dir() or blocks is None:
    if not LABELED_JSON.exists():
        raise FileNotFoundError(
            f"No output found at {LABELED_JSON}. Run the pipeline first (Cell 2)."
        )
    blocks = json.loads(LABELED_JSON.read_text(encoding="utf-8"))
    print(f"Loaded {len(blocks)} blocks from {LABELED_JSON}")

# Solid saturated badge colors — white text, visible on projector
LABEL_COLOR = {
    "title":               "#b45309",   # amber
    "section.title":       "#1d4ed8",   # blue
    "section.description": "#6d28d9",   # purple
    "question.text":       "#047857",   # green
    "answer.text":         "#374151",   # slate
}
# Left-border accent per label (for the row)
ROW_ACCENT = {
    "title":               "#f59e0b",
    "section.title":       "#3b82f6",
    "section.description": "#8b5cf6",
    "question.text":       "#10b981",
    "answer.text":         "#e5e7eb",
}

FONT = "font-family:'Inter','Segoe UI',system-ui,Arial,sans-serif;"

def badge(label):
    bg = LABEL_COLOR.get(label, "#374151")
    return (
        f'<span style="{FONT}background:{bg};color:#fff;'
        f'padding:4px 10px;border-radius:12px;font-size:12px;'
        f'font-weight:700;white-space:nowrap;letter-spacing:0.3px;">'
        f'{label}</span>'
    )

rows_html = ""
for i, b in enumerate(blocks):
    label  = b.get("label", "answer.text")
    conf   = b.get("confidence", 1.0)
    text   = b["text"].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    accent = ROW_ACCENT.get(label, "#e5e7eb")
    bg     = "#ffffff" if i % 2 == 0 else "#f8fafc"
    rows_html += (
        f'<tr style="background:{bg};border-left:4px solid {accent};">'
        f'<td style="padding:10px 12px;color:#64748b;font-size:13px;font-weight:600;white-space:nowrap;">{b["page"]}</td>'
        f'<td style="padding:10px 12px;">{badge(label)}</td>'
        f'<td style="padding:10px 14px;font-size:15px;color:#0f172a;line-height:1.5;">{text}</td>'
        f'<td style="padding:10px 12px;font-size:13px;color:#64748b;font-weight:600;text-align:right;white-space:nowrap;">{conf:.0%}</td>'
        f'</tr>'
    )

counts  = Counter(b.get("label", "answer.text") for b in blocks)
summary = "  ".join(
    f'<span style="margin-right:4px;">{badge(l)}'
    f'<span style="{FONT}margin-left:5px;font-size:14px;font-weight:700;color:#0f172a;">{n}</span></span>'
    for l, n in counts.most_common()
)

html = f"""
<div style="{FONT}max-width:980px;">
  <div style="margin-bottom:14px;padding:12px 16px;background:#f1f5f9;border-radius:10px;
              border:1px solid #cbd5e1;display:flex;align-items:center;flex-wrap:wrap;gap:8px;">
    <span style="font-size:13px;color:#475569;font-weight:700;margin-right:6px;text-transform:uppercase;letter-spacing:0.5px;">Summary</span>
    {summary}
  </div>
  <div style="border:2px solid #cbd5e1;border-radius:10px;overflow:hidden;max-height:560px;overflow-y:auto;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
    <table style="width:100%;border-collapse:collapse;">
      <thead>
        <tr style="background:#1e293b;position:sticky;top:0;z-index:1;">
          <th style="padding:12px 12px;text-align:left;font-size:12px;color:#94a3b8;font-weight:700;letter-spacing:0.5px;text-transform:uppercase;border-bottom:2px solid #334155;">PG</th>
          <th style="padding:12px 12px;text-align:left;font-size:12px;color:#94a3b8;font-weight:700;letter-spacing:0.5px;text-transform:uppercase;border-bottom:2px solid #334155;">LABEL</th>
          <th style="padding:12px 14px;text-align:left;font-size:12px;color:#94a3b8;font-weight:700;letter-spacing:0.5px;text-transform:uppercase;border-bottom:2px solid #334155;">TEXT</th>
          <th style="padding:12px 12px;text-align:right;font-size:12px;color:#94a3b8;font-weight:700;letter-spacing:0.5px;text-transform:uppercase;border-bottom:2px solid #334155;">CONF</th>
        </tr>
      </thead>
      <tbody>{rows_html}</tbody>
    </table>
  </div>
</div>
"""

HTML(html)

## 4 — Structured DMP output
The final nested structure — title, sections, questions, and answers — ready for the DMP Tool.

In [ ]:
import json
from IPython.display import HTML

# Load from disk — works whether or not the pipeline ran this session
if not STRUCTURED_JSON.exists():
    raise FileNotFoundError(
        f"No output found at {STRUCTURED_JSON}. Run the pipeline first (Cell 2)."
    )
structured = json.loads(STRUCTURED_JSON.read_text(encoding="utf-8"))
template   = structured["narrative"]["template"]

FONT = "font-family:'Inter','Segoe UI',system-ui,Arial,sans-serif;"

def esc(s):
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")

# ── Document title ────────────────────────────────────────────────────────────
doc_title  = template.get("title", "").strip()
title_html = (
    f'<div style="{FONT}background:#0f172a;color:#f8fafc;padding:18px 22px;border-radius:10px;'
    f'margin-bottom:18px;font-size:20px;font-weight:800;letter-spacing:-0.3px;">{esc(doc_title)}</div>'
    if doc_title else
    f'<div style="{FONT}background:#e2e8f0;color:#94a3b8;padding:14px 18px;border-radius:10px;'
    f'margin-bottom:18px;font-size:14px;font-style:italic;">No document title detected</div>'
)

# ── Stats bar ─────────────────────────────────────────────────────────────────
n_sections  = len(template.get("section", []))
n_questions = sum(len(s.get("question", [])) for s in template.get("section", []))
stats_html  = (
    f'<div style="{FONT}margin-bottom:18px;padding:10px 16px;background:#dbeafe;border-radius:8px;'
    f'font-size:14px;color:#1e40af;font-weight:600;">'
    f'<b>{n_sections}</b> sections &nbsp;·&nbsp; <b>{n_questions}</b> questions'
    f'&nbsp;&nbsp;<span style="color:#64748b;font-weight:400;">|&nbsp; Model: {MODEL} &nbsp;· Extractor: {EXTRACTOR}</span>'
    f'</div>'
)

# ── Sections ──────────────────────────────────────────────────────────────────
sections_html = ""
for sec in template.get("section", []):
    sec_title = esc(sec.get("title", "").strip())
    desc      = sec.get("description", "").strip()

    desc_html = (
        f'<div style="{FONT}margin:10px 0 14px 0;padding:10px 14px;background:#f5f3ff;'
        f'border-left:4px solid #6d28d9;border-radius:0 8px 8px 0;'
        f'font-size:14px;color:#4c1d95;font-style:italic;line-height:1.6;">{esc(desc)}</div>'
        if desc else ""
    )

    questions_html = ""
    for q in sec.get("question", []):
        q_text = esc(q.get("text", "").strip())
        ans    = q.get("answer", {}).get("json", {}).get("answer", "").strip()
        ans_html = (
            f'<div style="{FONT}margin-top:8px;padding:10px 14px;background:#f0fdf4;'
            f'border-left:4px solid #059669;border-radius:0 8px 8px 0;'
            f'font-size:15px;color:#1e293b;line-height:1.6;">{esc(ans)}</div>'
            if ans else
            f'<div style="{FONT}margin-top:6px;font-size:13px;color:#94a3b8;font-style:italic;">No answer captured</div>'
        )
        q_badge = (
            f'<span style="{FONT}display:inline-block;background:#047857;color:#fff;'
            f'padding:3px 10px;border-radius:10px;font-size:12px;font-weight:700;margin-bottom:6px;">'
            f'Q{q["order"]}</span>'
        )
        questions_html += (
            f'<div style="margin:12px 0;padding:12px 16px;background:#fafafa;'
            f'border:1px solid #e2e8f0;border-radius:8px;">'
            f'{q_badge}'
            f'<div style="{FONT}font-size:15px;color:#0f172a;font-weight:600;line-height:1.5;">{q_text}</div>'
            f'{ans_html}'
            f'</div>'
        )

    if not questions_html:
        questions_html = f'<div style="{FONT}font-size:13px;color:#94a3b8;font-style:italic;padding:6px 0;">No questions</div>'

    sections_html += (
        f'<div style="margin-bottom:20px;border:2px solid #bfdbfe;border-radius:10px;overflow:hidden;'
        f'box-shadow:0 1px 6px rgba(0,0,0,0.06);">'
        f'<div style="{FONT}background:#1d4ed8;color:#fff;padding:12px 18px;font-weight:700;font-size:16px;'
        f'letter-spacing:-0.2px;">Section {sec["order"]}: {sec_title}</div>'
        f'<div style="padding:14px 18px;background:#fff;">{desc_html}{questions_html}</div>'
        f'</div>'
    )

HTML(
    f'<div style="{FONT}max-width:920px;">'
    f'{title_html}{stats_html}{sections_html}</div>'
)